
# Hera CLI Summary and Dynamic Toolkit System

This notebook documents the command-line interfaces (CLIs) of the **Hera** framework:

- **`hera-toolkit`** — manage dynamic toolkits (list, load, register, import from JSON, default repository).
- **`hera-project`** — manage projects, repositories, versions, and list project measurements.

It also summarizes what we implemented:
- A **dynamic** toolkit registration & loading flow via JSON and repository-backed documents.
- New helper command **`hera-project project measurements list`** to inspect registered data sources (e.g., toolkits and experiments).



## Dynamic Toolkit Architecture (What We Built)

Historically, toolkits were available only via a **static in-code registry**. We added a **dynamic path**:
1. **JSON import** defines toolkits (with classpaths) and experiments (with data files).
2. These are **registered** into the project's repository as standard documents (e.g., type `ToolkitDataSource`).
3. `hera-toolkit list` now displays **both**: static (internal) and dynamic (database) toolkits.
4. `hera-toolkit load` can instantiate a toolkit by name, first checking the static registry, then the DB.
5. `hera-project project measurements list` shows a table of project data sources (toolkits, experiments, repository configs).

**Benefits:**
- Toolkits are discoverable and reproducible from JSON.
- Experiments (raw data) can be bulk-loaded in the same pass.
- CLI becomes user-friendly for non-developers.



## `hera-toolkit` — Manage Dynamic Toolkits

### 1) `list`
**Purpose:** Show all available toolkits for a project (internal + dynamic from DB/repository).  
**Usage:**
```bash
hera-toolkit list --project <PROJECT_NAME>
```
**Output Columns:**
- `toolkit`: toolkit identifier
- `cls`: Python classpath
- `source`: `internal` or `db`
- `type`: category (e.g., `measurements` / `simulations` / `riskassessment`)
- `repository`: repository name (for db-sourced toolkits)
- `version`: registered version tuple (for db-sourced toolkits)

**Example:**
```bash
!hera-toolkit list --project testDoc
```



### 2) `load`
**Purpose:** Instantiate a toolkit by name.  
It resolves in this order: static dict → DB document → optional auto-register helper (if implemented).

**Usage:**
```bash
hera-toolkit load --project <PROJECT_NAME> --name <TOOLKIT_NAME>
```

**Examples:**
```bash
!hera-toolkit load --project testDoc --name GIS_Raster_Topography
!hera-toolkit load --project testDoc --name GIS_Demography
```



### 3) `register`
**Purpose:** Register a toolkit **dynamically** using a Python classpath, so it becomes available from the DB.

**Usage:**
```bash
hera-toolkit register --project <PROJECT_NAME> --cls <pkg.mod.Class> --name <TOOLKIT_NAME> [--repository <REPO>] [--version 0,0,1]
```

**Notes:**
- `--cls` is a fully qualified Python classpath (e.g., `hera.measurements.GIS.raster.topography.TopographyToolkit`).
- The command stores a `ToolkitDataSource` document in the project repository.



### 4) `default-repository`
**Purpose:** Get or set the project's default repository name for toolkits.

**Usage:**
```bash
hera-toolkit default-repository show --project <PROJECT_NAME>
hera-toolkit default-repository set --project <PROJECT_NAME> --repository <REPO_NAME>
```



### 5) `import-json`
**Purpose:** Import toolkits and experiments from a JSON repository descriptor and register them dynamically.

**Usage:**
```bash
hera-toolkit import-json --project <PROJECT_NAME> --file </full/path/to/Repository.json> [--no-experiments]
```

**Behavior:**
- Registers toolkits as `ToolkitDataSource` documents (with `repository`, `version`, `classpath`, `parameters`).
- Optionally loads experiments (e.g., `Experiment_rawData`) if present in the JSON and `--no-experiments` is **not** set.

**Example:**
```bash
!hera-toolkit import-json --project testDoc --file /home/ilay/hera/hera/tests/Documentation_Repository.json
```



## `hera-project` — Manage Projects, Repositories, Versions, Measurements

### Database Management
```bash
hera-project db list
hera-project db create <connectionName> --username <USER> --password <PASS> --IP <HOST> --databaseName <DB>
hera-project db remove <connectionName>
```

### Project Management
```bash
hera-project project list
hera-project project create <PROJECT_NAME> [--directory <DIR>] [--noRepositories] [--dont-overwrite]
hera-project project dump <PROJECT_NAME> [--fileName <OUT>] [--format table|json] [query...]
hera-project project load <PROJECT_NAME> <dump_file.json>
hera-project project updateRepositories --projectName <PROJECT_NAME> [--overwrite]
```

### Versions
```bash
hera-project project version display <PROJECT_NAME> [--datasource <NAME>] [--default]
hera-project project version update <PROJECT_NAME> <DATASOURCE> <d,d,d>
```

### Repositories
```bash
hera-project repository list
hera-project repository add </full/path/to/Repository.json> [--overwrite]
hera-project repository remove <REPOSITORY_NAME>
hera-project repository show <REPOSITORY_NAME>
hera-project repository load <REPOSITORY_NAME> <PROJECT_NAME> [--overwrite]
```

### Measurements (New)
```bash
hera-project project measurements list --project <PROJECT_NAME> [--type <TYPE>] [--contains <SUBSTR>]
```
**Common types:** `ToolkitDataSource`, `Experiment_rawData`, `RepositoryConfig`  
**Columns explained:** `type`, `datasourceName`, `resource`, `dataFormat`, `version`, `repository`



## End-to-End Example Workflow

> This section demonstrates a typical flow to create a project, import toolkits, list them, load one, and inspect measurements.

```bash
# 1) Create a project (once)
!hera-project project create testDoc

# 2) Set default repository for the project
!hera-toolkit default-repository set --project testDoc --repository defaultRepo

# 3) Import toolkits + experiments from JSON
!hera-toolkit import-json --project testDoc --file /home/ilay/hera/hera/tests/Documentation_Repository.json

# 4) List toolkits (internal + db)
!hera-toolkit list --project testDoc

# 5) Load a toolkit by name
!hera-toolkit load --project testDoc --name GIS_Raster_Topography

# 6) Inspect project measurements
!hera-project project measurements list --project testDoc

# Examples of filtering:
!hera-project project measurements list --project testDoc --type ToolkitDataSource
!hera-project project measurements list --project testDoc --contains Topography
```



## Summary

- **Dynamic registration**: Toolkits and experiments can be declared in JSON and registered to the project's repository.
- **Discovery**: `hera-toolkit list` exposes both static and dynamic toolkits to users.
- **Instantiation**: `hera-toolkit load` creates toolkit instances by name (DB-first fallback logic implemented in `ToolkitHome`).
- **Inspection**: `hera-project project measurements list` gives a transparent view into what's actually stored in the project's data layer.

This notebook serves as **living documentation** for your CLI, enabling both developers and operators to use Hera confidently.
